# Random Forest Ablation: Remove Primary Mouse Hepatocytes

This notebook retrains the frozen Random Forest baseline after removing the primary mouse hepatocyte context and then reevaluates the model with the same outer-test workflow.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns

project_root = Path.cwd().resolve()
while not (project_root / "utils").exists() and project_root != project_root.parent:
    project_root = project_root.parent

if not (project_root / "utils").exists():
    raise RuntimeError("Could not locate project root containing the utils package")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from classical_ml.random_forest import (
    RandomForestDataBuilder,
    RandomForestExperimentConfig,
    RandomForestHoldoutSplitter,
    RandomForestTrainer,
    compute_regression_metrics,
)
from utils.merge_historic_data import load_merged_dataset

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)


## Frozen Configuration

This uses the same strict-cleaning, add-mRNA, no-normalized-conditions RF setup and the same frozen Optuna hyperparameters as the final baseline notebook.

In [2]:
cmsirna_path = os.environ.get("CMSIRNA_RAW_DATA_PATH")
historic_path = os.environ.get("CMSIRNA_RAW_HISTORIC_DATA_PATH")

assert cmsirna_path, "CMSIRNA_RAW_DATA_PATH is not set"
assert historic_path, "CMSIRNA_RAW_HISTORIC_DATA_PATH is not set"

config = RandomForestExperimentConfig(
    target_column="Inhibition",
    strict_cleaning=True,
    add_mrna=True,
    use_normalized_conditions=False,
    outer_test_size=0.33,
    n_splits=3,
    leak_n=30,
    random_state=42,
    n_estimators=500,
    max_depth=None,
    min_samples_split=7,
    min_samples_leaf=1,
    max_features=0.3,
    n_jobs=-1,
)

baseline_metrics = pd.DataFrame([{'pearson': 0.287733, 'spearman': 0.335756, 'rmse': 36.207205, 'mae': 28.724077}])
config


RandomForestExperimentConfig(target_column='Inhibition', target_len=25, strict_cleaning=True, add_mrna=True, fetch_missing_mrna=True, use_normalized_conditions=False, n_splits=3, leak_n=30, outer_test_size=0.33, random_state=42, n_estimators=500, max_depth=None, min_samples_split=7, min_samples_leaf=1, max_features=0.3, n_jobs=-1, tuning_n_iter=20, tuning_scoring='neg_mean_absolute_error', optuna_n_trials=20)

## What This Notebook Removes

Removed rows: `Cell_Type == "Primary mouse hepatocytes"`.

Why: this cell context looked especially suspicious because it was very poor on the final outer test, with very high MAE and weak Spearman, but it looked easy in the all-data fit. That pattern is consistent with a domain-shift or poor-transfer regime rather than stable signal.

In [3]:
builder = RandomForestDataBuilder(config)
merged_df = load_merged_dataset(cmsirna_path, historic_path)
enriched_df = builder.pipeline.enrich_dataset_with_encodings(
    merged_df,
    strict_cleaning=config.strict_cleaning,
    add_mrna=config.add_mrna,
)

removed_cell_type = "Primary mouse hepatocytes"
keep_mask = enriched_df["Cell_Type"] != removed_cell_type
removed_enriched_df = enriched_df.loc[~keep_mask].copy()
filtered_enriched_df = enriched_df.loc[keep_mask].reset_index(drop=True)

X, groups, y = builder.pipeline.prepare_for_classical_ml(
    filtered_enriched_df,
    target_column=config.target_column,
    use_normalized_conditions=config.use_normalized_conditions,
)

removal_summary = pd.DataFrame([{
    "starting_rows": int(len(enriched_df)),
    "rows_removed": int(len(removed_enriched_df)),
    "rows_remaining": int(len(filtered_enriched_df)),
    "pct_removed": float(len(removed_enriched_df) / len(enriched_df)),
    "removed_cell_type": removed_cell_type,
}])

removal_summary


loaded 3515 historic rows
merged 43153 CMsiRNA and 3515 historic rows into 46668
Running qc and data cleaning
dropped 4233 rows for in-vivo readings
dropped 565 rows for mM readings
dropped 115 rows for unknown or unwanted cell lines
dropped 47 rows for out-of-range inhibition
dropped 1749 rows for missing or unknown concentration
dropped 796 rows for concentration > 200 nM
filled 7522 rows for missing time of administration
dropped 2198 rows with a missing or >25 nt strand
dropped 6 columns: ['Modification_locations_Sense_strand', 'Modification_locations_Antisense_strand', 'Modifications_sense_strand', 'Modifications_AntiSense_strand_3_5', 'position_Antisense_strand', 'position_Sense_strand']
Mapping mRNA structural profiles
Error reading reference dataset: [Errno 2] No such file or directory: '/home/larsena8/software/fennec/src/fennec/support_files/train_data_v1.1.0_N=27742.csv'
Loaded 47 gene sequences from local cache
Loaded 0 gene sequences from reference CSV
Building gene -> mRNA

,starting_rows,rows_removed,rows_remaining,pct_removed,removed_cell_type
0,36965,501,36464,0.013553,Primary mouse hepatocytes


## Refit And Evaluate

The model is retrained from scratch after the ablation and then evaluated again with the same outer-test holdout workflow.

In [4]:
holdout_splitter = RandomForestHoldoutSplitter(config)
train_idx, test_idx = holdout_splitter.split(X, y, groups)

X_train = X[train_idx]
y_train = y[train_idx]
groups_train = groups[train_idx]
X_test = X[test_idx]
y_test = y[test_idx]
groups_test = groups[test_idx]

trainer = RandomForestTrainer(config)
trainer.fit(X_train, y_train)
y_test_pred = trainer.predict(X_test)

metric_values = compute_regression_metrics(y_test, y_test_pred)
ablation_metrics = pd.DataFrame([metric_values])
split_summary = pd.DataFrame([{
    "n_train": int(len(train_idx)),
    "n_test": int(len(test_idx)),
    "n_train_groups": int(len(np.unique(groups_train))),
    "n_test_groups": int(len(np.unique(groups_test))),
}])

test_metadata = filtered_enriched_df.iloc[test_idx].reset_index().rename(columns={"index": "source_index"}).copy()
test_metadata["patent_group"] = test_metadata.get("patent_ID", pd.Series(index=test_metadata.index, dtype=object)).fillna("HISTORIC_OR_UNKNOWN")
test_predictions = test_metadata.copy()
test_predictions["row_index"] = test_idx
test_predictions["group"] = groups_test
test_predictions["y_true"] = y_test
test_predictions["y_pred"] = y_test_pred
test_predictions["residual"] = test_predictions["y_true"] - test_predictions["y_pred"]
test_predictions["abs_error"] = test_predictions["residual"].abs()

split_summary


,n_train,n_test,n_train_groups,n_test_groups
0,25780,10684,36,18


## Baseline Comparison

These tables compare the original final-test baseline metrics against the retrained ablation run, then show the raw metric deltas.

In [5]:
comparison = pd.concat(
    [baseline_metrics.assign(run="baseline_final_test"), ablation_metrics.assign(run="ablation")],
    ignore_index=True,
)
comparison


,pearson,spearman,rmse,mae,run
0,0.287733,0.335756,36.207205,28.724077,baseline_final_test
1,0.303375,0.349720,35.080837,28.031715,ablation


In [6]:
delta_vs_baseline = ablation_metrics.iloc[0] - baseline_metrics.iloc[0]
delta_df = pd.DataFrame({
    "metric": delta_vs_baseline.index,
    "delta_vs_baseline": delta_vs_baseline.values,
})
delta_df


,metric,delta_vs_baseline
0,pearson,0.015642
1,spearman,0.013964
2,rmse,-1.126368
3,mae,-0.692362


## Test Prediction Preview

In [7]:
test_predictions[[
    "group",
    "Cell_Type",
    "Concentration_nM",
    "Time_of_administration_h",
    "patent_group",
    "y_true",
    "y_pred",
    "residual",
    "abs_error",
]].head(20)


,group,Cell_Type,Concentration_nM,Time_of_administration_h,patent_group,y_true,y_pred,residual,abs_error
0,CTNNB1,Hela,100.0,48.0,CN107365771B,88.00,57.280363,30.719637,30.719637
1,CTNNB1,Hela,100.0,48.0,CN107365771B,90.00,72.811090,17.188910,17.188910
2,CTNNB1,Hela,100.0,48.0,CN107365771B,90.00,60.169396,29.830604,29.830604
3,CTNNB1,Hela,100.0,48.0,CN107365771B,89.00,76.476882,12.523118,12.523118
4,CTNNB1,Hela,100.0,48.0,CN107365771B,87.00,73.758111,13.241889,13.241889
5,CTNNB1,Hela,100.0,48.0,CN107365771B,91.00,76.943176,14.056824,14.056824
6,CTNNB1,Hela,100.0,48.0,CN107365771B,62.00,57.925782,4.074218,4.074218
7,CTNNB1,Hep3B,10.0,24.0,WO2023003995A8,93.62,65.977308,27.642692,27.642692
8,CTNNB1,Hep3B,10.0,24.0,WO2023003995A8,92.55,53.759403,38.790597,38.790597
9,CTNNB1,Hep3B,10.0,24.0,WO2023003995A8,93.48,73.827669,19.652331,19.652331


## Notes To Fill In After Running

- Did Spearman improve or drop relative to baseline?
- Did MAE and RMSE improve enough to justify the data removal?
- Does this ablation improve the model by removing noise, or does it simply reduce coverage?